# 01 — Preparação, anonimização e tratamento dos dados

Este notebook transforma o arquivo bruto do formulário em uma base **anonimizada, limpa, padronizada e pronta para análise**.

Fluxo:

`base_original.csv` → validação → consentimento → anonimização → limpeza → padronização → ausentes → tipagem/ordem → múltipla escolha → variáveis derivadas → validação final → `dados_processados.csv`



## 1. Configuração e importações

In [1]:
from pathlib import Path
import re
import unicodedata
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

ARQUIVO_ENTRADA = Path("dados_form.csv")
ARQUIVO_SAIDA = Path("dados_processados.csv")
ARQUIVO_DICIONARIO = Path("dicionario_dados.csv")

## 2. Carregamento da base bruta



In [2]:
if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ARQUIVO_ENTRADA.resolve()}\n"
        "No Colab, faça upload de 'dados_form.csv' ou ajuste ARQUIVO_ENTRADA."
    )

dados_brutos = pd.read_csv(ARQUIVO_ENTRADA)

print("Dimensão bruta:", dados_brutos.shape)
print("Linhas duplicadas exatas:", dados_brutos.duplicated().sum())
print("Quantidade de colunas:", dados_brutos.shape[1])

Dimensão bruta: (13, 31)
Linhas duplicadas exatas: 0
Quantidade de colunas: 31


## 3. Padronização inicial dos nomes das colunas

In [3]:
dados = dados_brutos.copy()

dados.columns = (
    dados.columns.astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

for i, coluna in enumerate(dados.columns):
    print(i, repr(coluna))

0 'Carimbo de data/hora'
1 'Nome de usuário'
2 'TERMO DE CONSENTIMENTO LIVRE E ESCLARECIDO (TCLE) Declaro que li, ou que alguém leu para mim, as informações contidas neste documento antes de aceitar participar da pesquisa. Declaro que a linguagem utilizada foi clara e que todas as minhas dúvidas foram esclarecidas satisfatoriamente. Confirmo que recebi uma cópia deste Termo de Consentimento Livre e Esclarecido. Compreendo que minha participação é voluntária e que posso desistir a qualquer momento, sem qualquer penalidade. Declaro ter 18 anos ou mais e dou meu consentimento de livre e espontânea vontade para participar deste estudo.'
3 '1) Nome Completo:'
4 '2) WhatsApp para contato:'
5 '3) Seu papel principal:'
6 '4) Nível de atuação:'
7 '5) Tempo de experiência na área:'
8 '6) Principais tipos de ocorrência com que você lida:'
9 '7) Hoje, como as ocorrências chegam até você?'
10 '8) Quais são os principais problemas atuais (marque até 3):'
11 '9) Em uma escala 1– 5, qual a gravidade d

## 4. Validação do consentimento (TCLE)



In [4]:
colunas_tcle = [
    c for c in dados.columns
    if "TERMO DE CONSENTIMENTO LIVRE E ESCLARECIDO" in c.upper()
]

if len(colunas_tcle) > 1:
    raise ValueError(f"Mais de uma coluna de TCLE encontrada: {colunas_tcle}")

coluna_tcle = colunas_tcle[0] if colunas_tcle else None

if coluna_tcle is None:
    print("AVISO: coluna de TCLE não localizada.")
else:
    print("Respostas do TCLE:")
    display(dados[coluna_tcle].value_counts(dropna=False))

    respostas_tcle = dados[coluna_tcle].astype("string").str.strip()

    padrao_recusa = (
        r"\bn[aã]o\s+(aceito|concordo|autorizo|consinto|quero participar)"
        r"|\bdiscordo\b|\bn[aã]o desejo participar\b"
    )

    recusas = respostas_tcle.str.contains(
        padrao_recusa,
        case=False,
        na=False,
        regex=True
    )

    if recusas.any():
        raise ValueError(
            f"Foram encontradas {int(recusas.sum())} resposta(s) que parecem recusa ao TCLE. "
            "Revise e exclua esses registros antes de continuar."
        )

Respostas do TCLE:


,count
"TERMO DE CONSENTIMENTO LIVRE E ESCLARECIDO (TCLE) Declaro que li, ou que alguém leu para mim, as informações contidas neste documento antes de aceitar participar da pesquisa. Declaro que a linguagem utilizada foi clara e que todas as minhas dúvidas foram esclarecidas satisfatoriamente. Confirmo que recebi uma cópia deste Termo de Consentimento Livre e Esclarecido. Compreendo que minha participação é voluntária e que posso desistir a qualquer momento, sem qualquer penalidade. Declaro ter 18 anos ou mais e dou meu consentimento de livre e espontânea vontade para participar deste estudo.",
Li e concordo em participar da pesquisa,13


/tmp/ipykernel_1186/1966284900.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  recusas = respostas_tcle.str.contains(


## 5. Anonimização



In [5]:
padroes_sensiveis = [
    "Carimbo de data/hora",
    "Nome de usuário",
    "TERMO DE CONSENTIMENTO LIVRE E ESCLARECIDO",
    "1) Nome Completo",
    "2) WhatsApp para contato",
]

colunas_sensiveis = [
    c for c in dados.columns
    if any(p.lower() in c.lower() for p in padroes_sensiveis)
]

print("Colunas removidas:")
for c in colunas_sensiveis:
    print("-", c[:150])

dados = dados.drop(columns=colunas_sensiveis, errors="ignore")

Colunas removidas:
- Carimbo de data/hora
- Nome de usuário
- TERMO DE CONSENTIMENTO LIVRE E ESCLARECIDO (TCLE) Declaro que li, ou que alguém leu para mim, as informações contidas neste documento antes de aceitar
- 1) Nome Completo:
- 2) WhatsApp para contato:


## 6. Renomeação das perguntas


In [6]:
nomes_por_numero = {
    3: "funcao",
    4: "nivel",
    5: "tempo_exp",
    6: "tipos_ocorrencia",
    7: "notificacao",
    8: "desafios",
    9: "grau_fp",
    10: "dados_registro",
    11: "tempo_resp",
    12: "internet",
    13: "cadastro",
    14: "ver_ocorrencias_proximas",
    15: "telas",
    16: "filtros_mapa",
    17: "regra_confirmado",
    18: "responsavel_validacao",
    19: "fontes_dados",
    20: "horizonte_prev",
    21: "usabilidade",
    22: "usa_modelo",
    23: "verificacao",
    24: "explicacao",
    25: "funcionalidades",
    26: "riscos",
    27: "grau_util",
    28: "aceite_teste",
}

renomear = {}

for coluna in dados.columns:
    match = re.match(r"^(\d+)\)\s*", coluna)
    if match:
        numero = int(match.group(1))
        if numero in nomes_por_numero:
            renomear[coluna] = nomes_por_numero[numero]

dados = dados.rename(columns=renomear)

colunas_esperadas = list(nomes_por_numero.values())
faltantes = [c for c in colunas_esperadas if c not in dados.columns]
extras = [c for c in dados.columns if c not in colunas_esperadas]

if faltantes:
    raise ValueError(f"Colunas esperadas não foram encontradas após a renomeação: {faltantes}")

if extras:
    print("AVISO — colunas adicionais não mapeadas:", extras)

print("Colunas analíticas:", dados.columns.tolist())

Colunas analíticas: ['funcao', 'nivel', 'tempo_exp', 'tipos_ocorrencia', 'notificacao', 'desafios', 'grau_fp', 'dados_registro', 'tempo_resp', 'internet', 'cadastro', 'ver_ocorrencias_proximas', 'telas', 'filtros_mapa', 'regra_confirmado', 'responsavel_validacao', 'fontes_dados', 'horizonte_prev', 'usabilidade', 'usa_modelo', 'verificacao', 'explicacao', 'funcionalidades', 'riscos', 'grau_util', 'aceite_teste']


## 7. Cópia de trabalho e limpeza textual geral

In [7]:
dados_limpos = dados.copy()

# Strings vazias passam a ser valores ausentes.
for col in dados_limpos.select_dtypes(include="object").columns:
    dados_limpos[col] = (
        dados_limpos[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace(r"^\s*$", pd.NA, regex=True)
    )

## 8. Padronização de categorias comprovadamente equivalentes


In [8]:
dados_limpos["funcao"] = dados_limpos["funcao"].replace({
    "Bombeira Militar": "Bombeiro(a) Militar",
    "Bombeiro militar": "Bombeiro(a) Militar",
    "Bombeiro Militar": "Bombeiro(a) Militar",
    "Tenho experiência": "Resposta não classificável",
})

dados_limpos["usa_modelo"] = dados_limpos["usa_modelo"].replace({
    "não": "Não",
    "NÃO": "Não",
})

dados_limpos["tipos_ocorrencia"] = (
    dados_limpos["tipos_ocorrencia"]
    .str.replace("Queda de arvores", "Queda de árvores", regex=False)
)

dados_limpos["dados_registro"] = (
    dados_limpos["dados_registro"]
    .str.replace("Texto Livre", "Texto livre", regex=False)
)

## 9. Auditoria de valores ausentes e duplicados


In [9]:
auditoria = pd.DataFrame({
    "dtype": dados_limpos.dtypes.astype(str),
    "n_nulos": dados_limpos.isna().sum(),
    "perc_nulos": (dados_limpos.isna().mean() * 100).round(2),
    "n_unicos": dados_limpos.nunique(dropna=True),
})

print("Dimensão:", dados_limpos.shape)
print("Duplicados exatos:", dados_limpos.duplicated().sum())
display(auditoria.sort_values(["n_nulos", "n_unicos"], ascending=[False, True]))

Dimensão: (13, 26)
Duplicados exatos: 0


,dtype,n_nulos,perc_nulos,n_unicos
usa_modelo,string,5,38.46,4
tempo_resp,string,0,0.00,2
internet,string,0,0.00,2
aceite_teste,string,0,0.00,2
nivel,string,0,0.00,3
explicacao,string,0,0.00,3
tempo_exp,string,0,0.00,4
cadastro,string,0,0.00,4
ver_ocorrencias_proximas,string,0,0.00,4
regra_confirmado,string,0,0.00,4


## 10. Tipagem e codificação ordinal


In [10]:
mapas_ordinais = {
    "tempo_exp": {
        "< 1 ano": 0,
        "1 - 3": 1,
        "4 - 7": 2,
        "8+": 3,
    },
    "tempo_resp": {
        "Imediato (<10s)": 0,
        "até 5 min": 1,
    },
    "internet": {
        "Não": 0,
        "Às vezes": 1,
        "Sim": 2,
    },
}

for coluna, mapa in mapas_ordinais.items():
    nova_coluna = f"{coluna}_ord"
    dados_limpos[nova_coluna] = dados_limpos[coluna].map(mapa).astype("Int64")

    inesperados = (
        dados_limpos.loc[
            dados_limpos[coluna].notna() & dados_limpos[nova_coluna].isna(),
            coluna
        ]
        .drop_duplicates()
        .tolist()
    )

    if inesperados:
        raise ValueError(
            f"Valores não reconhecidos em {coluna}: {inesperados}. "
            "Atualize o mapa ordinal antes de continuar."
        )

for coluna in ["grau_fp", "grau_util"]:
    dados_limpos[coluna] = pd.to_numeric(
        dados_limpos[coluna],
        errors="raise"
    ).astype("Int64")

    invalidos = ~dados_limpos[coluna].dropna().between(1, 5)
    if invalidos.any():
        raise ValueError(f"{coluna} possui valores fora da escala 1–5.")

## 11. Tratamento da pergunta sobre uso de modelo


In [11]:
dados_limpos["usa_modelo_status"] = dados_limpos["usa_modelo"].apply(
    lambda x:
        pd.NA if pd.isna(x)
        else "Não" if x == "Não"
        else "Uso informado"
).astype("string")

dados_limpos["modelo_informado"] = (
    dados_limpos["usa_modelo"]
    .where(
        dados_limpos["usa_modelo"].notna()
        & (dados_limpos["usa_modelo"] != "Não")
    )
    .astype("string")
)

display(
    dados_limpos[
        ["usa_modelo", "usa_modelo_status", "modelo_informado"]
    ]
)

,usa_modelo,usa_modelo_status,modelo_informado
0,Não,Não,<NA>
1,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>
3,Climatempo,Uso informado,Climatempo
4,Não,Não,<NA>
5,Não,Não,<NA>
6,<NA>,<NA>,<NA>
7,Defesa civil,Uso informado,Defesa civil
8,Não,Não,<NA>
9,Sim! Defesa Civil do Estado,Uso informado,Sim! Defesa Civil do Estado


## 12. Questões de múltipla escolha


In [12]:
colunas_multiplas = [
    "tipos_ocorrencia",
    "notificacao",
    "desafios",
    "dados_registro",
    "telas",
    "filtros_mapa",
    "responsavel_validacao",
    "fontes_dados",
    "horizonte_prev",
    "usabilidade",
    "verificacao",
]

def slug(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    return texto.strip("_")

opcoes_multiplas = {}

for coluna in colunas_multiplas:
    series_lista = (
        dados_limpos[coluna]
        .fillna("")
        .apply(
            lambda x: [
                item.strip()
                for item in str(x).split(";")
                if item.strip()
            ]
        )
    )

    opcoes = sorted({
        item
        for lista in series_lista
        for item in lista
    })

    opcoes_multiplas[coluna] = opcoes

    for opcao in opcoes:
        nome_indicador = f"{coluna}__{slug(opcao)}"
        dados_limpos[nome_indicador] = series_lista.apply(
            lambda itens: int(opcao in itens)
        ).astype("Int64")

print("Indicadores criados:", sum(len(v) for v in opcoes_multiplas.values()))

Indicadores criados: 72


## 13. Respostas abertas

In [13]:
colunas_abertas = ["funcionalidades", "riscos"]

for coluna in colunas_abertas:
    dados_limpos[coluna] = (
        dados_limpos[coluna]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

display(dados_limpos[colunas_abertas])

,funcionalidades,riscos
0,"Segurança, transparência e estudos técnicos","Erro, informações incorretas e nervosismo"
1,"Informação precisa, imagem visual da ocorrência, prazo previsto","Informação errada, falta de conhecimento técnico, escassez de recursos para solução do problema"
2,Prevenção de risco,Pânico
3,"Confiança, Eficiência e Design","Erro, desinformação, layout não amigável"
4,"Facilidade de uso, rapidez e desempenho.","Erro, spam e vazamento de dados."
5,"Modelo, mapa, informações confiáveis","Pânico, magnitude, demora na resposta"
6,"Registro de dados, imagens e equipe",Pânico
7,"Comunicação, logística e recursos humanos.","Falta de equipamento, desconhecimento da ocorrência e falha humana."
8,Identificação do risco de desastre com atencendecia de no mínimo 3 horas antes.,A população precisa receber o alerta de forma que não ocasione medo e nem pânico
9,Acessibilidade,Falta de acesso


## 14. Dicionário de dados

In [14]:
classificacao = {
    "funcao": ("qualitativa", "nominal"),
    "nivel": ("qualitativa", "nominal"),
    "tempo_exp": ("qualitativa", "ordinal"),
    "tipos_ocorrencia": ("qualitativa", "nominal múltipla"),
    "notificacao": ("qualitativa", "nominal múltipla"),
    "desafios": ("qualitativa", "nominal múltipla"),
    "grau_fp": ("qualitativa", "ordinal Likert"),
    "dados_registro": ("qualitativa", "nominal múltipla"),
    "tempo_resp": ("qualitativa", "ordinal"),
    "internet": ("qualitativa", "ordinal"),
    "cadastro": ("qualitativa", "nominal"),
    "ver_ocorrencias_proximas": ("qualitativa", "nominal"),
    "telas": ("qualitativa", "nominal múltipla"),
    "filtros_mapa": ("qualitativa", "nominal múltipla"),
    "regra_confirmado": ("qualitativa", "nominal"),
    "responsavel_validacao": ("qualitativa", "nominal múltipla"),
    "fontes_dados": ("qualitativa", "nominal múltipla"),
    "horizonte_prev": ("qualitativa", "nominal múltipla"),
    "usabilidade": ("qualitativa", "nominal múltipla"),
    "usa_modelo": ("qualitativa", "nominal + texto aberto"),
    "verificacao": ("qualitativa", "nominal múltipla"),
    "explicacao": ("qualitativa", "nominal"),
    "funcionalidades": ("qualitativa", "texto aberto"),
    "riscos": ("qualitativa", "texto aberto"),
    "grau_util": ("qualitativa", "ordinal Likert"),
    "aceite_teste": ("qualitativa", "nominal binária"),
}

registros_dicionario = []

for coluna, (natureza, subtipo) in classificacao.items():
    registros_dicionario.append({
        "variavel": coluna,
        "natureza": natureza,
        "subtipo": subtipo,
        "derivada": False,
        "origem": coluna,
    })

for coluna in mapas_ordinais:
    registros_dicionario.append({
        "variavel": f"{coluna}_ord",
        "natureza": "quantitativa codificada",
        "subtipo": "ordem auxiliar",
        "derivada": True,
        "origem": coluna,
    })

registros_dicionario.extend([
    {
        "variavel": "usa_modelo_status",
        "natureza": "qualitativa",
        "subtipo": "nominal derivada",
        "derivada": True,
        "origem": "usa_modelo",
    },
    {
        "variavel": "modelo_informado",
        "natureza": "qualitativa",
        "subtipo": "texto informado",
        "derivada": True,
        "origem": "usa_modelo",
    },
])

for coluna, opcoes in opcoes_multiplas.items():
    for opcao in opcoes:
        registros_dicionario.append({
            "variavel": f"{coluna}__{slug(opcao)}",
            "natureza": "quantitativa binária",
            "subtipo": "indicador 0/1 de múltipla escolha",
            "derivada": True,
            "origem": coluna,
        })

dicionario_dados = pd.DataFrame(registros_dicionario)
display(dicionario_dados.head(30))

,variavel,natureza,subtipo,derivada,origem
0,funcao,qualitativa,nominal,False,funcao
1,nivel,qualitativa,nominal,False,nivel
2,tempo_exp,qualitativa,ordinal,False,tempo_exp
3,tipos_ocorrencia,qualitativa,nominal múltipla,False,tipos_ocorrencia
4,notificacao,qualitativa,nominal múltipla,False,notificacao
5,desafios,qualitativa,nominal múltipla,False,desafios
6,grau_fp,qualitativa,ordinal Likert,False,grau_fp
7,dados_registro,qualitativa,nominal múltipla,False,dados_registro
8,tempo_resp,qualitativa,ordinal,False,tempo_resp
9,internet,qualitativa,ordinal,False,internet


## 15. Validação final



In [15]:
# 1) Nenhuma coluna sensível deve permanecer.
sensiveis_restantes = [
    c for c in dados_limpos.columns
    if any(p.lower() in c.lower() for p in padroes_sensiveis)
]
assert not sensiveis_restantes, (
    f"Colunas sensíveis ainda presentes: {sensiveis_restantes}"
)

# 2) As 26 variáveis principais devem existir.
faltantes_final = [
    c for c in colunas_esperadas
    if c not in dados_limpos.columns
]
assert not faltantes_final, (
    f"Variáveis principais ausentes: {faltantes_final}"
)

# 3) Os indicadores de múltipla escolha devem conter somente 0, 1 ou NA.
colunas_indicadoras = [
    c for c in dados_limpos.columns
    if "__" in c
]

for coluna in colunas_indicadoras:
    valores = set(dados_limpos[coluna].dropna().unique().tolist())
    assert valores.issubset({0, 1}), (
        f"{coluna} possui valores inválidos: {valores}"
    )

print("Dimensão final:", dados_limpos.shape)
print("Duplicados exatos:", dados_limpos.duplicated().sum())
print("Total de valores ausentes:", int(dados_limpos.isna().sum().sum()))

display(
    pd.DataFrame({
        "n_nulos": dados_limpos.isna().sum(),
        "perc_nulos": (dados_limpos.isna().mean() * 100).round(2),
    })
    .query("n_nulos > 0")
    .sort_values("n_nulos", ascending=False)
)

print("Validação final concluída.")

Dimensão final: (13, 103)
Duplicados exatos: 0
Total de valores ausentes: 20


,n_nulos,perc_nulos
modelo_informado,10,76.92
usa_modelo,5,38.46
usa_modelo_status,5,38.46


Validação final concluída.


## 16. Exportação


In [16]:
dados_limpos.to_csv(
    ARQUIVO_SAIDA,
    index=False,
    encoding="utf-8-sig"
)

dicionario_dados.to_csv(
    ARQUIVO_DICIONARIO,
    index=False,
    encoding="utf-8-sig"
)

print(f"Base processada: {ARQUIVO_SAIDA.resolve()}")
print(f"Dicionário:      {ARQUIVO_DICIONARIO.resolve()}")
print("Linhas:", len(dados_limpos))
print("Colunas:", dados_limpos.shape[1])

Base processada: /content/dados_processados.csv
Dicionário:      /content/dicionario_dados.csv
Linhas: 13
Colunas: 103
